In [ ]:
from pathlib import Path
import json

# Find the root directory, no matter where we are. 
def find_project_root(marker="README.md"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root()

path = PROJECT_ROOT / "data/processed/redirect_urls.json"
print(path)

In [ ]:

with open(path, "r") as f:
    redirect_urls = json.load(f)

In [ ]:
import pandas as pd 
urls = pd.DataFrame(pd.Series(redirect_urls), columns=["redirect_url"])

In [ ]:
len(urls)

In [ ]:
urls = urls[urls["redirect_url"].str.contains("details")]

In [ ]:
len(urls)

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import random

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-AU,en;q=0.5",
}

def get_full_description(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        scripts = soup.find_all('script', type='application/ld+json')
        
        for script in scripts:
            data = json.loads(script.string)
            if isinstance(data, list):
                data = data[0]
            if 'description' in data:
                return data.get('description')
        
        return None
    
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
# Iterate through the data frame of URLs, and attach the description in the description column. 
import time
import random
import json
import os
from datetime import datetime

progress_file = "data/progress_tracking/scrape_progress.json"
save_interval = 50

# Resume from progress if it exists
start_index = 0
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        progress = json.load(f)
    start_index = progress["last_processed"]
    print(f"Resuming from URL #{start_index + 1}")

total = len(urls)
start_time = datetime.now()

for i, (index, row) in enumerate(urls.iterrows()):
    if i < start_index:
        continue

    url = row["redirect_url"]
    description = get_full_description(url)
    urls.at[index, "description"] = description

    status = "ok" if description else "no description"
    print(f"[{i+1}/{total}] {status}: {url[:70]}")

    # Save checkpoint
    if (i + 1) % save_interval == 0:
        urls.to_csv("data/progress_tracking/urls_with_descriptions.csv", index=False)
        with open(progress_file, "w") as f:
            json.dump({"last_processed": i + 1, "timestamp": datetime.now().isoformat()}, f)

        elapsed = datetime.now() - start_time
        avg = elapsed.total_seconds() / (i + 1 - start_index)
        remaining = (total - i - 1) * avg / 3600
        print(f"Saved. {elapsed} elapsed, ~{remaining:.1f}h remaining")

    if i + 1 < total:
        time.sleep(random.uniform(2, 3))

# Final save
urls.to_csv("data/raw/urls_with_descriptions.csv", index=False)
with open(progress_file, "w") as f:
    json.dump({"last_processed": total, "timestamp": datetime.now().isoformat()}, f)
print(f"Done. {total} URLs processed.")

In [ ]:
urls.iloc[0]["redirect_url"]

In [ ]:
# print response.text as nicely formatted html with linebreaks.
# Force print the whole thing without abreviating it.

print(BeautifulSoup(response.text, 'html.parser').prettify())

In [ ]:
# Get the 'description' field from the html using tags, not the function
soup = BeautifulSoup(response.text, 'html.parser')
script = soup.find('script', type='application/ld+json')

# Extract the description field from the script
data = json.loads(script.string)
if isinstance(data, list):
    data = data[0]
print(data)